# MongoDB Intermediate: How to Design Schemas & Search Data

## Courses & Instructors

**Difficulty: Intermediate | ~40 min | Requires Labs 1–3**

*Lab 4 of 7 in the MongoDB Mastery series.*

In this lab, you will learn schema design patterns — embedding vs. referencing — and full-text search in MongoDB.

You will learn how to:
1. Connect to a MongoDB Atlas cluster
2. Populate two related collections: `courses` and `instructors`
3. Embed bounded data (schedule) inside the course document
4. Reference shared data (instructors) and join with `$lookup`
5. Create a text index and search courses by keyword
6. Generate a formatted summary report

In [ ]:
!pip install -qU "pymongo[srv,tls]==4.10.1" python-dotenv==1.0.1 certifi

This installs the MongoDB Python driver (`pymongo`) with `srv` and `tls` extras, `python-dotenv` for loading credentials, and `certifi` for up-to-date CA certificates.

### Step 1 — Connect to MongoDB

In [1]:
import os
import certifi
from dotenv import load_dotenv
import pymongo

# Load the Atlas connection string from the .env file one directory up
load_dotenv("../.env")
uri = os.environ["MONGODB_URI"]

# Connect to the real MongoDB Atlas cluster
client = pymongo.MongoClient(uri, tlsCAFile=certifi.where())

# Access the database and collections
db = client["school_db"]
courses = db["courses"]
instructors = db["instructors"]

print("Connected to MongoDB Atlas")

Connected to MongoDB Atlas


Same connection pattern as Labs 1–3 — `load_dotenv` reads from the shared `.env` file, and `certifi.where()` provides trusted CA certificates for SSL.

### Step 2 — Populate Both Collections

In [2]:
courses.drop()
instructors.drop()

# Insert 3 instructors into the instructors collection
instructor_records = [
    {"instructor_id": "INS001", "name": "Dr. Sarah Thorne",  "department": "Computer Science", "bio": "Specializes in algorithms and data structures with 15 years of teaching experience."},
    {"instructor_id": "INS002", "name": "Prof. James Reed",  "department": "Mathematics",      "bio": "Research focus on applied statistics and linear algebra in machine learning."},
    {"instructor_id": "INS003", "name": "Dr. Maria Chen",    "department": "Physics",           "bio": "Expert in computational physics and quantum mechanics simulation."},
]
instructors.insert_many(instructor_records)
print(f"Inserted {len(instructor_records)} instructors.")

# Insert 6 courses with embedded schedule subdocuments and instructor references
course_records = [
    {"course_id": "CS101",   "title": "Introduction to Computer Science",
     "description": "Fundamentals of programming, algorithms, and computational thinking",
     "instructor_id": "INS001", "seats": 120,
     "schedule": {"meeting_times": "MWF 9:00-9:50", "location": "Hamilton Hall Room 204"}},
    {"course_id": "CS201",   "title": "Data Structures and Algorithms",
     "description": "Advanced data structures including trees, graphs, and algorithm analysis",
     "instructor_id": "INS001", "seats": 80,
     "schedule": {"meeting_times": "TTh 10:30-11:45", "location": "Science Building Room 105"}},
    {"course_id": "MATH201", "title": "Linear Algebra",
     "description": "Vectors, matrices, eigenvalues, and applications to data science",
     "instructor_id": "INS002", "seats": 100,
     "schedule": {"meeting_times": "MWF 11:00-11:50", "location": "Math Building Room 301"}},
    {"course_id": "MATH301", "title": "Probability and Statistics",
     "description": "Probability distributions, hypothesis testing, and statistical inference",
     "instructor_id": "INS002", "seats": 75,
     "schedule": {"meeting_times": "TTh 1:00-2:15", "location": "Math Building Room 205"}},
    {"course_id": "PHYS101", "title": "General Physics I",
     "description": "Classical mechanics, thermodynamics, and wave phenomena",
     "instructor_id": "INS003", "seats": 90,
     "schedule": {"meeting_times": "MWF 2:00-2:50", "location": "Physics Lab Room 110"}},
    {"course_id": "PHYS301", "title": "Computational Physics",
     "description": "Numerical methods, scientific computing, and algorithm simulation",
     "instructor_id": "INS003", "seats": 45,
     "schedule": {"meeting_times": "TTh 3:30-4:45", "location": "Physics Lab Room 202"}},
]
result = courses.insert_many(course_records)
print(f"Inserted {len(result.inserted_ids)} courses.")

Inserted 3 instructors.
Inserted 6 courses.


Each course stores a `schedule` subdocument (embedded) and an `instructor_id` (reference). Both collections are dropped first to ensure a clean slate on re-runs.

### Step 3 — Demonstrate Embedded Schedule

In [3]:
# Read the embedded schedule directly from the course document — no join needed
cs101 = courses.find_one({"course_id": "CS101"}, {"_id": 0})

print(f"--- Embedded Schedule for {cs101['course_id']}: {cs101['title']} ---")
print(f"  Meeting times: {cs101['schedule']['meeting_times']}")
print(f"  Location: {cs101['schedule']['location']}")

--- Embedded Schedule for CS101: Introduction to Computer Science ---
  Meeting times: MWF 9:00-9:50
  Location: Hamilton Hall Room 204


The schedule data lives inside the course document. A single `find()` returns everything — the course metadata and its schedule — with no second query or join required. This is the embedding pattern at work: bounded, always-read-together data stored alongside the main record.

### Step 4 — $lookup: Join Courses to Instructors

In [4]:
pipeline = [
    {"$lookup": {
        "from": "instructors",
        "localField": "instructor_id",
        "foreignField": "instructor_id",
        "as": "instructor_info"
    }},
    {"$unwind": "$instructor_info"},
    {"$project": {
        "_id": 0,
        "course_id": 1,
        "title": 1,
        "instructor_name": "$instructor_info.name",
        "department": "$instructor_info.department"
    }}
]

results = list(courses.aggregate(pipeline))

print(f"--- Course Instructor Lookup ({len(results)} results) ---")
for doc in results:
    print(f"{doc['course_id']}: {doc['title']} -> {doc['instructor_name']} ({doc['department']})")

--- Course Instructor Lookup (6 results) ---
CS101: Introduction to Computer Science -> Dr. Sarah Thorne (Computer Science)
CS201: Data Structures and Algorithms -> Dr. Sarah Thorne (Computer Science)
MATH201: Linear Algebra -> Prof. James Reed (Mathematics)
MATH301: Probability and Statistics -> Prof. James Reed (Mathematics)
PHYS101: General Physics I -> Dr. Maria Chen (Physics)
PHYS301: Computational Physics -> Dr. Maria Chen (Physics)


`$lookup` joins each course to its instructor by matching `instructor_id`. `$unwind` flattens the resulting array into a single subdocument. `$project` reshapes the output to show only the fields we care about — course ID, title, instructor name, and department.

### Step 5 — Text Index and Search

In [5]:
# Create a text index on the description field
courses.create_index([("description", "text")])
print("Text index created on 'description' field.")

# Search for courses containing the word 'algorithms'
search_results = list(courses.find(
    {"$text": {"$search": "algorithms"}},
    {"_id": 0, "course_id": 1, "title": 1, "description": 1}
))

print(f"\n--- Text Search: 'algorithms' ({len(search_results)} results) ---")
for doc in search_results:
    print(f"{doc['course_id']}: {doc['title']}")

Text index created on 'description' field.

--- Text Search: 'algorithms' (3 results) ---
CS101: Introduction to Computer Science
PHYS301: Computational Physics
CS201: Data Structures and Algorithms


The text index lets MongoDB search the *content* of the `description` field. The query `{"$text": {"$search": "algorithms"}}` matches any course whose description contains that word or a stemmed variant — here, CS101 ("algorithms"), CS201 ("Algorithms"), and PHYS301 ("algorithm") all match due to MongoDB's built-in stemming.

### Step 6 — Summary Report

In [6]:
total_courses = courses.count_documents({})
total_instructors = instructors.count_documents({})

# Average seats across all courses
seats_pipeline = [{"$group": {"_id": None, "avg_seats": {"$avg": "$seats"}}}]
avg_seats = list(courses.aggregate(seats_pipeline))[0]["avg_seats"]

# Instructor with most courses taught
teach_pipeline = [
    {"$group": {"_id": "$instructor_id", "count": {"$sum": 1}}},
    {"$sort": {"count": -1, "_id": 1}},
    {"$limit": 1}
]
top = list(courses.aggregate(teach_pipeline))[0]
top_name = instructors.find_one({"instructor_id": top["_id"]})["name"]

print("       COURSES & INSTRUCTORS")
print(f"\nTotal courses: {total_courses}")
print(f"Total instructors: {total_instructors}")
print(f"Average seats per course: {avg_seats:.1f}")
print(f"\nTop instructor by courses: {top_name} ({top['count']} courses)")

# Re-run text search for the report
search_results = list(courses.find(
    {"$text": {"$search": "algorithms"}},
    {"_id": 0, "course_id": 1, "title": 1}
))
print(f"\n--- Text Search: 'algorithms' ({len(search_results)} results) ---")
for doc in search_results:
    print(f"{doc['course_id']}: {doc['title']}")

       COURSES & INSTRUCTORS

Total courses: 6
Total instructors: 3
Average seats per course: 85.0

Top instructor by courses: Dr. Sarah Thorne (2 courses)

--- Text Search: 'algorithms' (3 results) ---
CS101: Introduction to Computer Science
PHYS301: Computational Physics
CS201: Data Structures and Algorithms


Collects the key metrics from both collections into one formatted report — course counts, seat statistics, the instructor with the most courses, and a text search for "algorithms." Note: all three instructors teach exactly 2 courses each — the tie is broken alphabetically by `instructor_id`, so the output shows Dr. Sarah Thorne (INS001) first.